# Word embeddings

_Deep Learning_

**Turn words into number-vectors where similar words land near each other.**

Computers need numbers, not words. So we turn each word into a vector.

     The naive way is one-hot: a long vector that is all 0 except a single 1. But one-hot vectors say nothing about meaning; "cat" and "dog" look as unrelated as "cat" and "car".

---

This notebook is a practice scaffold for the **Word embeddings** lesson. We'll build a lookup table of learnable word vectors, then see that real vectors cluster by meaning. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Create an embedding table

`nn.Embedding` is just a lookup table: one learnable vector per word in the vocabulary. Here we ask for 1000 words, each represented by a 16-dimensional vector. At the start these vectors are random — training is what gives them meaning.

In [ ]:
import torch
import torch.nn as nn

vocab_size = 1000
dim = 16
embed = nn.Embedding(vocab_size, dim)    # 1000 words, each a 16-d vector

### Step 2 — Look up a sentence's word vectors

We represent a sentence as a list of integer word ids, then pass it through the table. The lookup replaces each id with its row, turning a `(1, 3)` batch of ids into a `(1, 3, 16)` batch of vectors — one 16-d vector per word.

In [ ]:
# a sentence as word ids: "the cat sat" -> [4, 17, 92]
sentence = torch.tensor([[4, 17, 92]])
vectors = embed(sentence)                # look up each word's vector

### Step 3 — Inspect the looked-up vectors

The shapes confirm the lookup, and we peek at the first four numbers of the "cat" vector (the middle word, index 1). These numbers are ordinary network **parameters**: during training backprop nudges the vectors of words that appear in similar contexts closer together.

In [ ]:
print("input ids :", tuple(sentence.shape))   # (1, 3)
print("embeddings:", tuple(vectors.shape))     # (1, 3, 16)
print("'cat' vector[:4]:", vectors[0, 1, :4].round(decimals=3).tolist())
# these vectors are parameters: backprop nudges similar words together

## Visualize the data & results

_Do real word vectors place related words (royalty, places, animals) near each other?_

### Step 1 — Hand-build vectors for a few words

To see clustering without training, we hand-pick 4-dimensional vectors for 13 words across three themes — royalty/people, places, and animals. Words we expect to be related are given deliberately similar coordinates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

words = {
 "king": [0.9, 0.95, 0.1, 0.8], "queen": [0.88, 0.1, 0.12, 0.82],
 "prince": [0.85, 0.9, 0.15, 0.7], "princess": [0.83, 0.12, 0.18, 0.72],
 "man": [0.2, 0.92, 0.3, 0.1], "woman": [0.18, 0.08, 0.32, 0.12],
 "paris": [-0.8, 0.4, 0.9, -0.6], "france": [-0.85, 0.42, 0.95, -0.55],
 "london": [-0.78, 0.45, 0.88, -0.5], "england": [-0.82, 0.46, 0.92, -0.48],
 "cat": [0.1, -0.85, -0.7, 0.3], "dog": [0.12, -0.82, -0.72, 0.28],
 "kitten": [0.08, -0.88, -0.68, 0.32]}

### Step 2 — Project the vectors down to 2-D with PCA

The vectors live in 4-D, which we can't plot directly. PCA finds the two directions that capture the most variation and projects every word onto them, giving each word an (x, y) point we can draw while preserving most of the structure.

In [ ]:
names = list(words)
vectors = np.array(list(words.values()))
P = PCA(n_components=2, random_state=0).fit_transform(vectors)

### Step 3 — Plot the words coloured by theme

We colour each word by its theme and scatter the 2-D points. Related words land near each other — royalty together, places together, animals together — which is exactly the property that makes embeddings useful: distance in vector space reflects similarity in meaning.

In [ ]:
groups = {"royalty/people": names[:6], "places": names[6:10], "animals": names[10:]}
colors = {"royalty/people": "#4ea1ff", "places": "#7ee787", "animals": "#ffb454"}
for group, members in groups.items():
    pts = np.array([P[names.index(w)] for w in members])
    plt.scatter(pts[:, 0], pts[:, 1], color=colors[group], label=group)
plt.title("Real word vectors projected to 2-D (PCA): meaning clusters")
plt.xlabel("component 1")
plt.ylabel("component 2")
plt.legend()
plt.show()